In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load dataset
df = pd.read_csv(
    "../dataset/StudentPerformanceFactors.csv"
)

# Separate features and target
X = df.drop("Exam_Score", axis=1)
y = df["Exam_Score"]

# Identify numerical and categorical features
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

# Recreate training and testing split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Dataset loaded successfully!")
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Numerical features:", len(numerical_columns))
print("Categorical features:", len(categorical_columns))

Dataset loaded successfully!
Training shape: (5285, 19)
Testing shape: (1322, 19)
Numerical features: 6
Categorical features: 13


In [3]:
# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_columns),
    ("cat", categorical_transformer, categorical_columns)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [4]:

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor

models = {
    "Ridge Regression": Ridge(alpha=10),

    "ElasticNet": ElasticNet(
        alpha=0.01,
        l1_ratio=0.5,
        max_iter=10000
    ),

    "Support Vector Regression": SVR(
        kernel="rbf",
        C=10,
        epsilon=0.1
    ),

    "Extra Trees Regressor": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "KNN Regressor": KNeighborsRegressor(
        n_neighbors=5,
        weights="distance"
    )
}

print("Five regression models defined successfully!")


Five regression models defined successfully!


In [5]:

results = []

for name, model in models.items():

    print(f"Evaluating: {name}")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "MAE": -scores["test_mae"].mean(),
        "RMSE": -scores["test_rmse"].mean(),
        "R2": scores["test_r2"].mean()
    })

print("\nAll model evaluations completed!")


Evaluating: Ridge Regression
Evaluating: ElasticNet
Evaluating: Support Vector Regression
Evaluating: Extra Trees Regressor
Evaluating: KNN Regressor

All model evaluations completed!


In [6]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE",
    ascending=True
).reset_index(drop=True)

display(results_df.round(6))

,Model,MAE,RMSE,R2
0,Ridge Regression,0.507010,2.052375,0.723229
1,ElasticNet,0.510964,2.052802,0.723111
2,Support Vector Regression,0.537448,2.078413,0.716403
3,Extra Trees Regressor,1.116501,2.357604,0.637352
4,KNN Regressor,1.552792,2.719930,0.518184


In [7]:

from sklearn.model_selection import GridSearchCV

# Ridge Regression pipeline
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

# Test different regularization strengths
param_grid = {
    "model__alpha": [
        0.001, 0.01, 0.1, 1, 5, 10,
        20, 50, 100, 200, 500, 1000
    ]
}

# Perform grid search using 5-fold CV
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    return_train_score=True
)

ridge_search.fit(X_train, y_train)

print("Best Ridge Parameters:")
print(ridge_search.best_params_)

print("\nBest Cross-validation MAE:")
print(-ridge_search.best_score_)


Best Ridge Parameters:
{'model__alpha': 10}

Best Cross-validation MAE:
0.5070104864057161


In [8]:
ridge_results = pd.DataFrame(
    ridge_search.cv_results_
)

ridge_results = ridge_results[
    [
        "param_model__alpha",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
]

ridge_results["Mean_Test_MAE"] = (
    -ridge_results["mean_test_score"]
)

ridge_results["Mean_Train_MAE"] = (
    -ridge_results["mean_train_score"]
)

ridge_results = ridge_results.sort_values(
    "Mean_Test_MAE"
)

display(
    ridge_results[
        [
            "param_model__alpha",
            "Mean_Test_MAE",
            "Mean_Train_MAE",
            "std_test_score"
        ]
    ].round(6)
)

,param_model__alpha,Mean_Test_MAE,Mean_Train_MAE,std_test_score
5,10.000,0.507010,0.503490,0.062345
4,5.000,0.507026,0.503456,0.062261
3,1.000,0.507112,0.503503,0.062197
2,0.100,0.507143,0.503522,0.062181
1,0.010,0.507147,0.503524,0.062179
0,0.001,0.507147,0.503524,0.062179
6,20.000,0.507305,0.503905,0.062516
7,50.000,0.510669,0.507597,0.063060
8,100.000,0.523556,0.520714,0.063090
9,200.000,0.566871,0.564396,0.063735


In [9]:
# Get the best Ridge pipeline
final_model = ridge_search.best_estimator_

# Fit the selected model on the complete training dataset
final_model.fit(X_train, y_train)

# Save the final model
joblib.dump(
    final_model,
    "../models/best_model.pkl"
)

print("Final optimized model saved successfully!")
print("Selected model: Ridge Regression")
print("Best alpha:", ridge_search.best_params_["model__alpha"])

Final optimized model saved successfully!
Selected model: Ridge Regression
Best alpha: 10


In [10]:
# Load the saved model
verified_model = joblib.load(
    "../models/best_model.pkl"
)

print("Model verification successful!")
print("Model:", verified_model.named_steps["model"])

Model verification successful!
Model: Ridge(alpha=10)
